In [1]:
# ============================================
# COMPLETE BASEBALL WIN PREDICTION MODEL
# WITH ADVANCED METRICS AND OPTIMIZED HUBER
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_validate, GridSearchCV

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set up data directory
DATA_DIR = Path(os.environ.get("LOCAL_DATA_DIR", Path.cwd() / "data")).resolve()
print(f"Loading datasets from {DATA_DIR}")

# Load data
data_df = pd.read_csv(DATA_DIR / "data.csv")
predict_df = pd.read_csv(DATA_DIR / "predict.csv")
print(f"Data set shape: {data_df.shape}")
print(f"Predict set shape: {predict_df.shape}")

Loading datasets from /Users/simgsr/Documents/GitHub/NTU_M3_Kaggle_MoneyBall/data
Data set shape: (1812, 51)
Predict set shape: (453, 45)


In [2]:
# ============================================
# FUNCTION TO ADD ADVANCED BASEBALL METRICS
# ============================================

def add_advanced_metrics(df):
    """
    Add advanced baseball metrics including OBP, SLG, OPS, WHIP, and more
    """
    df_advanced = df.copy()
    
    # ========== OFFENSIVE METRICS ==========
    
    # Calculate Singles (1B) = Hits - (2B + 3B + HR)
    df_advanced['1B'] = df_advanced['H'] - (df_advanced['2B'] + df_advanced['3B'] + df_advanced['HR'])
    
    # On-Base Percentage (OBP) - Complete version
    if 'HBP' in df_advanced.columns and 'SF' in df_advanced.columns:
        df_advanced['OBP'] = (df_advanced['H'] + df_advanced['BB'] + df_advanced['HBP']) / \
                             (df_advanced['AB'] + df_advanced['BB'] + df_advanced['HBP'] + df_advanced['SF'])
    else:
        df_advanced['OBP'] = (df_advanced['H'] + df_advanced['BB']) / (df_advanced['AB'] + df_advanced['BB'])
    
    # Slugging Percentage (SLG)
    df_advanced['SLG'] = (df_advanced['1B'] + 2 * df_advanced['2B'] + 
                          3 * df_advanced['3B'] + 4 * df_advanced['HR']) / df_advanced['AB']
    
    # On-Base Plus Slugging (OPS)
    df_advanced['OPS'] = df_advanced['OBP'] + df_advanced['SLG']
    
    # Batting Average (BA)
    df_advanced['BA'] = df_advanced['H'] / df_advanced['AB']
    
    # Isolated Power (ISO) = SLG - BA
    df_advanced['ISO'] = df_advanced['SLG'] - df_advanced['BA']
    
    # Total Bases (TB)
    df_advanced['TB'] = df_advanced['1B'] + 2*df_advanced['2B'] + 3*df_advanced['3B'] + 4*df_advanced['HR']
    
    # Secondary Average (SEC) = (BB + (TB - H) + SB) / AB
    df_advanced['SEC'] = (df_advanced['BB'] + (df_advanced['TB'] - df_advanced['H']) + df_advanced['SB']) / df_advanced['AB']
    
    # Walk to Strikeout Ratio for batters (BB/SO)
    df_advanced['BB_SO_ratio'] = df_advanced['BB'] / df_advanced['SO'].replace(0, 1)
    
    # ========== PITCHING METRICS ==========
    
    # WHIP (Walks + Hits per Inning Pitched)
    df_advanced['IP'] = df_advanced['IPouts'] / 3
    df_advanced['WHIP'] = (df_advanced['BBA'] + df_advanced['HA']) / df_advanced['IP'].replace(0, 1)
    
    # Strikeout to Walk Ratio (K/BB) for pitchers
    df_advanced['K_BB_ratio'] = df_advanced['SOA'] / df_advanced['BBA'].replace(0, 1)
    
    # Calculated ERA to verify
    df_advanced['ERA_calc'] = (df_advanced['ER'] * 9) / df_advanced['IP'].replace(0, 1)
    
    # Fielding Independent Pitching (FIP) - simplified version
    df_advanced['FIP'] = (13 * df_advanced['HRA'] + 3 * df_advanced['BBA'] - 2 * df_advanced['SOA']) / df_advanced['IP'].replace(0, 1) + 3.2
    
    # ========== TEAM PERFORMANCE METRICS ==========
    
    # Run Differential (RD)
    df_advanced['RD'] = df_advanced['R'] - df_advanced['RA']
    
    # Pythagorean Expectation (expected win percentage)
    df_advanced['PyW_pct'] = (df_advanced['R']**2) / (df_advanced['R']**2 + df_advanced['RA']**2)
    df_advanced['PyW'] = df_advanced['PyW_pct'] * df_advanced['G']
    
    # Handle potential infinite values
    advanced_cols = ['OBP', 'SLG', 'OPS', 'WHIP', 'BA', 'ISO', 'SEC', 'BB_SO_ratio', 
                     'K_BB_ratio', 'ERA_calc', 'FIP', 'RD', 'PyW_pct', 'PyW', '1B', 'TB', 'IP']
    
    for col in advanced_cols:
        if col in df_advanced.columns:
            df_advanced[col] = df_advanced[col].replace([np.inf, -np.inf], np.nan)
            df_advanced[col] = df_advanced[col].fillna(df_advanced[col].median())
    
    return df_advanced

# Add advanced metrics
data_df = add_advanced_metrics(data_df)
predict_df = add_advanced_metrics(predict_df)
print("ADVANCED BASEBALL METRICS ADDED")

ADVANCED BASEBALL METRICS ADDED


In [3]:
# ============================================
# DEFINE FEATURE SET
# ============================================

# Default features
default_features = [
    'G', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB', 'CS', 'HBP', 'SF',
    'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA',
    'E', 'DP', 'FP', 'attendance', 'BPF', 'PPF',
    'R_per_game', 'RA_per_game', 'mlb_rpg',
    'era_1', 'era_2', 'era_3', 'era_4', 'era_5', 'era_6', 'era_7', 'era_8',
    'decade_1910', 'decade_1920', 'decade_1930', 'decade_1940', 'decade_1950',
    'decade_1960', 'decade_1970', 'decade_1980', 'decade_1990', 'decade_2000', 'decade_2010'
]

# Advanced metrics
advanced_metrics = ['1B', 'OBP', 'SLG', 'OPS', 'WHIP', 'BA', 'ISO', 'SEC', 
                   'BB_SO_ratio', 'K_BB_ratio', 'ERA_calc', 'FIP', 'RD', 'PyW_pct', 'PyW', 'TB', 'IP']

# Combine all features
all_features = default_features + advanced_metrics

# Filter available features
available_features = [col for col in all_features if col in data_df.columns and col in predict_df.columns]
print(f"\nTotal features available: {len(available_features)}")
print(f"  Default features: {len([c for c in default_features if c in available_features])}")
print(f"  Advanced metrics: {len([c for c in advanced_metrics if c in available_features])}")

# ============================================
# PREPARE DATA
# ============================================

X = data_df[available_features]
y = data_df['W']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
one_hot_cols = [col for col in X_train.columns if col.startswith(('era_', 'decade_'))]
other_cols = [col for col in X_train.columns if col not in one_hot_cols]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[other_cols] = scaler.fit_transform(X_train[other_cols])
X_test_scaled[other_cols] = scaler.transform(X_test[other_cols])

print(f"\nTraining set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")


Total features available: 61
  Default features: 44
  Advanced metrics: 17

Training set shape: (1449, 61)
Test set shape: (363, 61)


In [4]:
# ============================================
# PART 1: LINEAR REGRESSION BASELINE
# ============================================

print("\n" + "="*60)
print("PART 1: LINEAR REGRESSION BASELINE")
print("="*60)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_train_preds = lr.predict(X_train_scaled)
lr_test_preds = lr.predict(X_test_scaled)

lr_train_mae = mean_absolute_error(y_train, lr_train_preds)
lr_test_mae = mean_absolute_error(y_test, lr_test_preds)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_preds))
lr_test_r2 = r2_score(y_test, lr_test_preds)

print(f"\n{'='*30}")
print("LINEAR REGRESSION RESULTS")
print(f"{'='*30}")
print(f"Train MAE: {lr_train_mae:>10.4f}")
print(f"Test MAE:  {lr_test_mae:>10.4f}")
print(f"Test RMSE: {lr_test_rmse:>10.4f}")
print(f"Test R²:   {lr_test_r2:>10.4f}")
print(f"{'-'*30}")

# Mapping coefficients to feature names (assuming X_train is a DataFrame)
lr_coeff_df = pd.Series(lr.coef_, index=X_train.columns)
print("Top Coefficients:")
print(lr_coeff_df.sort_values(ascending=False))


PART 1: LINEAR REGRESSION BASELINE

LINEAR REGRESSION RESULTS
Train MAE:     2.6120
Test MAE:      2.8106
Test RMSE:     3.5310
Test R²:       0.9224
------------------------------
Top Coefficients:
R         1.253667e+13
ISO       8.355963e+11
H         6.241183e+11
OBP       6.053652e+11
IP        4.955479e+11
              ...     
1B       -4.210216e+11
IPouts   -4.955479e+11
OPS      -1.744110e+12
RA       -1.252165e+13
RD       -1.478646e+13
Length: 61, dtype: float64


In [5]:
# ============================================
# PART 2: HYPERPARAMETER TUNING FOR RIDGE AND LASSO
# ============================================

print("\n" + "="*60)
print("PART 2: RIDGE & LASSO TUNING")
print("="*60)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Lasso tuning
lasso_alphas = np.logspace(-4, 1, 13)
lasso_cv = LassoCV(alphas=lasso_alphas, cv=kfold, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)
print(f"Best Lasso alpha: {lasso_cv.alpha_:.6f}")
print(f"Features selected by Lasso: {np.sum(lasso_cv.coef_ != 0)}")

# Ridge tuning
ridge_alphas = np.logspace(-2, 1, 11)
ridge_cv = RidgeCV(alphas=ridge_alphas, scoring='neg_mean_absolute_error', cv=kfold)
ridge_cv.fit(X_train_scaled, y_train)
print(f"\nBest Ridge alpha: {ridge_cv.alpha_:.4f}")


# Train tuned models
best_ridge = Ridge(alpha=ridge_cv.alpha_)
best_ridge.fit(X_train_scaled, y_train)
ridge_preds = best_ridge.predict(X_test_scaled)
best_lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
best_lasso.fit(X_train_scaled, y_train)
lasso_preds = best_lasso.predict(X_test_scaled)

# Evaluate Lasso models
print(f"\nLasso Performance:")
print(f"  Train MAE: {mean_absolute_error(y_train, best_lasso.predict(X_train_scaled)):.4f}")
print(f"  Test MAE: {mean_absolute_error(y_test, lasso_preds):.4f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, lasso_preds)):.4f}")
print(f"  Test R²: {r2_score(y_test, lasso_preds):.4f}")

# Mapping coefficients to feature names (assuming X_train is a DataFrame)
lasso_coeff_df = pd.Series(lasso_cv.coef_, index=X_train.columns)
print("\nTop Coefficients:")
print(lasso_coeff_df.sort_values(ascending=False))

# Evaluate Ridge models 
print(f"\nRidge Performance:")
print(f"  Train MAE: {mean_absolute_error(y_train, best_ridge.predict(X_train_scaled)):.4f}")
print(f"  Test MAE: {mean_absolute_error(y_test, ridge_preds):.4f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, ridge_preds)):.4f}")
print(f"  Test R²: {r2_score(y_test, ridge_preds):.4f}")

# Mapping coefficients to feature names (assuming X_train is a DataFrame)
ridge_coeff_df = pd.Series(ridge_cv.coef_, index=X_train.columns)
print("\nTop Coefficients:")
print(ridge_coeff_df.sort_values(ascending=False))


PART 2: RIDGE & LASSO TUNING
Best Lasso alpha: 0.004642
Features selected by Lasso: 38

Best Ridge alpha: 2.5119

Lasso Performance:
  Train MAE: 2.6231
  Test MAE: 2.7986
  Test RMSE: 3.5230
  Test R²: 0.9227

Top Coefficients:
IPouts         4.887214
SV             4.081164
PyW_pct        3.823397
CG             3.272072
RD             2.106908
                 ...   
era_6         -0.549942
E             -0.617914
decade_1920   -1.848869
era_1         -2.550388
AB            -4.056994
Length: 61, dtype: float64

Ridge Performance:
  Train MAE: 2.6193
  Test MAE: 2.8008
  Test RMSE: 3.5273
  Test R²: 0.9225

Top Coefficients:
SV             4.058961
CG             3.223914
PyW            2.921809
PyW_pct        2.484692
R              2.344786
                 ...   
WHIP          -1.008382
decade_1910   -1.235596
era_1         -1.235596
decade_1920   -1.951275
AB            -3.495924
Length: 61, dtype: float64


In [6]:
# ============================================
# PART 3: OPTIMIZED HUBER REGRESSION
# ============================================

print("\n" + "="*60)
print("PART 3: OPTIMIZED HUBER REGRESSION")
print("="*60)

# Grid search for optimal Huber parameters
huber_params = {
    'epsilon': [1.0, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5, 3.0, 4.0, 5.0],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'max_iter': [1000, 2000]
}

huber = HuberRegressor()
huber_grid = GridSearchCV(
    huber, huber_params, cv=5, 
    scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1
)
huber_grid.fit(X_train_scaled, y_train)

print(f"\nBest Huber parameters:")
print(f"  epsilon: {huber_grid.best_params_['epsilon']}")
print(f"  alpha: {huber_grid.best_params_['alpha']}")
print(f"  max_iter: {huber_grid.best_params_['max_iter']}")

best_huber = huber_grid.best_estimator_
huber_train_preds = best_huber.predict(X_train_scaled)
huber_test_preds = best_huber.predict(X_test_scaled)

huber_train_mae = mean_absolute_error(y_train, huber_train_preds)
huber_test_mae = mean_absolute_error(y_test, huber_test_preds)
huber_test_rmse = np.sqrt(mean_squared_error(y_test, huber_test_preds))
huber_test_r2 = r2_score(y_test, huber_test_preds)

print(f"\nOptimized Huber Performance:")
print(f"  Train MAE: {huber_train_mae:.4f}")
print(f"  Test MAE: {huber_test_mae:.4f}")
print(f"  Test RMSE: {huber_test_rmse:.4f}")
print(f"  Test R²: {huber_test_r2:.4f}")



PART 3: OPTIMIZED HUBER REGRESSION
Fitting 5 folds for each of 80 candidates, totalling 400 fits


/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:342: ConvergenceWarning: lbfgs failed to c


Best Huber parameters:
  epsilon: 2.0
  alpha: 0.1
  max_iter: 1000

Optimized Huber Performance:
  Train MAE: 2.6110
  Test MAE: 2.8050
  Test RMSE: 3.5270
  Test R²: 0.9225


In [7]:
# ============================================
# PART 4: RANDOM FOREST AND GRADIENT BOOSTING
# ============================================

print("\n" + "="*60)
print("PART 4: TREE-BASED MODELS")
print("="*60)

# Random Forest tuning
print("\nTuning Random Forest...")
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [4, 8]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_grid = GridSearchCV(rf, rf_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1)
rf_grid.fit(X_train_scaled, y_train)

print(f"\nBest Random Forest parameters: {rf_grid.best_params_}")
best_rf = rf_grid.best_estimator_
rf_preds = best_rf.predict(X_test_scaled)

rf_test_mae = mean_absolute_error(y_test, rf_preds)
rf_test_r2 = r2_score(y_test, rf_preds)

# Gradient Boosting
print("\nTraining Gradient Boosting...")
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
gbr.fit(X_train_scaled, y_train)
gbr_preds = gbr.predict(X_test_scaled)

gbr_test_mae = mean_absolute_error(y_test, gbr_preds)
gbr_test_r2 = r2_score(y_test, gbr_preds)

print(f"\nRandom Forest Performance:")
print(f" train MAE: {mean_absolute_error(y_train, best_rf.predict(X_train_scaled)):.4f}")
print(f"  Test MAE: {rf_test_mae:.4f}, Test R²: {rf_test_r2:.4f}")
print(f"Gradient Boosting Performance:")
print(f"  Test MAE: {gbr_test_mae:.4f}, Test R²: {gbr_test_r2:.4f}")

# Mapping coefficients to feature names (assuming X_train is a DataFrame)
rf_coeff_df = pd.Series(best_rf.feature_importances_, index=X_train.columns)
print("Top Coefficients:")
print(rf_coeff_df.sort_values(ascending=False))



PART 4: TREE-BASED MODELS

Tuning Random Forest...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Best Random Forest parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}

Training Gradient Boosting...

Random Forest Performance:
 train MAE: 1.7540
  Test MAE: 3.2173, Test R²: 0.8959
Gradient Boosting Performance:
  Test MAE: 3.1870, Test R²: 0.8958
Top Coefficients:
PyW            0.908031
PyW_pct        0.016234
RD             0.015295
SV             0.010461
DP             0.002172
                 ...   
decade_1940    0.000009
era_2          0.000007
decade_2010    0.000001
era_8          0.000000
era_3          0.000000
Length: 61, dtype: float64


In [8]:
# ============================================
# PART 5: ENSEMBLE MODELS
# ============================================

print("\n" + "="*60)
print("PART 5: ENSEMBLE MODELS")
print("="*60)

# Voting Regressor
voting_model = VotingRegressor([
    ('lr', LinearRegression()),
    ('ridge', best_ridge),
    ('lasso', best_lasso),
    ('huber', best_huber),
])

voting_model.fit(X_train_scaled, y_train)
voting_preds = voting_model.predict(X_test_scaled)

voting_mae = mean_absolute_error(y_test, voting_preds)
voting_rmse = np.sqrt(mean_squared_error(y_test, voting_preds))
voting_r2 = r2_score(y_test, voting_preds)

print(f"\nVoting Regressor (Average):")
print(f"  Test MAE: {voting_mae:.4f}")
print(f"  Test RMSE: {voting_rmse:.4f}")
print(f"  Test R²: {voting_r2:.4f}")

# Stacking Regressor
stacking_model = StackingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('ridge', best_ridge),
        ('lasso', best_lasso),
        ('huber', best_huber),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stacking_model.fit(X_train_scaled, y_train)
stacking_preds = stacking_model.predict(X_test_scaled)

stacking_mae = mean_absolute_error(y_test, stacking_preds)
stacking_rmse = np.sqrt(mean_squared_error(y_test, stacking_preds))
stacking_r2 = r2_score(y_test, stacking_preds)

print(f"\nStacking Regressor (Meta-Learner):")
print(f"  Train MAE: {mean_absolute_error(y_train, stacking_model.predict(X_train_scaled)):.4f}")
print(f"  Test MAE: {stacking_mae:.4f}")
print(f"  Test RMSE: {stacking_rmse:.4f}")
print(f"  Test R²: {stacking_r2:.4f}")


PART 5: ENSEMBLE MODELS

Voting Regressor (Average):
  Test MAE: 2.8026
  Test RMSE: 3.5251
  Test R²: 0.9226

Stacking Regressor (Meta-Learner):
  Train MAE: 2.6354
  Test MAE: 2.7994
  Test RMSE: 3.5251
  Test R²: 0.9226


In [9]:
# ============================================
# PART 9: CROSS-VALIDATION RESULTS
# ============================================

print("\n" + "="*60)
print("PART 9: CROSS-VALIDATION RESULTS")
print("="*60)

# 5-fold cross-validation for top 3 models
top_models = {
    'Linear Regression': LinearRegression(),
    'lasso': best_lasso,
    'Ridge': best_ridge,
    'Optimized Huber': best_huber,
    'Stacking Ensemble': stacking_model
}

cv_results_list = []
for name, model in top_models.items():
    cv_mae = cross_val_score(model, X_train_scaled, y_train, 
                             cv=kfold, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_r2 = cross_val_score(model, X_train_scaled, y_train, 
                            cv=kfold, scoring='r2', n_jobs=-1)
    
    cv_results_list.append({
        'Model': name,
        'CV MAE': -cv_mae.mean(),
        'CV MAE Std': cv_mae.std(),
        'CV R²': cv_r2.mean(),
        'CV R² Std': cv_r2.std()
    })

cv_results_df = pd.DataFrame(cv_results_list).sort_values('CV MAE')
print(cv_results_df.to_string(index=False))




PART 9: CROSS-VALIDATION RESULTS
            Model   CV MAE  CV MAE Std    CV R²  CV R² Std
            lasso 2.716716    0.114908 0.933081   0.006250
            Ridge 2.718727    0.118335 0.932881   0.006408
Stacking Ensemble 2.722463    0.117085 0.932992   0.006120
  Optimized Huber 2.726784    0.123251 0.932379   0.006806
Linear Regression 2.733230    0.118773 0.931863   0.006826


In [10]:
# ============================================
# PART 10: GENERATING SUBMISSIONS (ALL MODELS)
# ============================================

print("\n" + "="*60)
print("PART 10: GENERATING SUBMISSIONS FOR ALL MODELS")
print("="*60)

# 1. Prepare prediction data (Features and Scaling)
# Ensure predict_df contains the same features used during training
predict_scaled = predict_df[available_features].copy()
predict_scaled[other_cols] = scaler.transform(predict_scaled[other_cols])

# 2. Iterate through all models in top_models
for name, model in top_models.items():
    print(f"\nProcessing Model: {name}")
    
    # Clean the name for a valid filename (replace spaces/special chars)
    file_safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    submission_path = f'./submission/submission_{file_safe_name}.csv'
    
    # 3. Retrain on the ENTIRE training set for maximum data usage
    model.fit(X_train_scaled, y_train)
    
    # 4. Make predictions
    preds = model.predict(predict_scaled)
    
    # 5. Create submission DataFrame
    # Rounding and clipping to 0-162 (Standard Baseball Season)
    sub_df = pd.DataFrame({
        'ID': predict_df['ID'],
        'W': np.round(preds).astype(int).clip(0, 162)
    })
    
    # 6. Save to CSV
    sub_df.to_csv(submission_path, index=False)
    
    # 7. Print Summary Statistics for this model
    print(f"  ✅ Saved to: {submission_path}")


PART 10: GENERATING SUBMISSIONS FOR ALL MODELS

Processing Model: Linear Regression
  ✅ Saved to: ./submission/submission_linear_regression.csv

Processing Model: lasso
  ✅ Saved to: ./submission/submission_lasso.csv

Processing Model: Ridge
  ✅ Saved to: ./submission/submission_ridge.csv

Processing Model: Optimized Huber
  ✅ Saved to: ./submission/submission_optimized_huber.csv

Processing Model: Stacking Ensemble
  ✅ Saved to: ./submission/submission_stacking_ensemble.csv
